# Практическое занятие №6
## Неконтролируемое обучение (Unsupervised Learning)

**Цель:** освоить базовый цикл неконтролируемого обучения: подготовка данных → кластеризация → оценка качества кластеров → визуализация → интерпретация результатов.

**Как работать:**
1. Скачайте свой CSV `dataset_variant_N.csv`.
2. Положите CSV в ту же папку, что и этот ноутбук.
3. В ячейке ниже установите `VARIANT = N`.
4. Выполните ячейки последовательно и заполните текстовые ответы (где отмечено).

> В этом ПЗ **нет** целевой переменной `target`. Мы ищем структуру данных и/или аномалии.


In [ ]:
# ================================
# ЯЧЕЙКА 1. Импорт библиотек
# ================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

# Для поиска аномалий (используем как доп. инструмент)
from sklearn.ensemble import IsolationForest



In [ ]:
# ================================
# ЯЧЕЙКА 2. Указание номера варианта и загрузка данных
# ================================
VARIANT = 1  # <-- УКАЖИТЕ НОМЕР СВОЕГО ВАРИАНТА (1–15)

filename = f"dataset_variant_{VARIANT}.csv"
df = pd.read_csv(filename)

print("Файл:", filename)
print("Размер данных:", df.shape)
df.head()


In [ ]:
# ================================
# ЯЧЕЙКА 3. Первичный анализ данных
# ================================
df.info()

print("
Пропуски по столбцам:")
print(df.isna().sum())

print("
Описательная статистика:")
df.describe()


## Ячейка 4. Формулировка задачи (заполнить вручную)

**В этой ячейке студент должен написать текстом:**
- какую структуру он ожидает увидеть (кластеры/аномалии/сложная форма кластеров);
- какие методы планирует применить (KMeans, DBSCAN, PCA, IsolationForest);
- как будет интерпретировать результат.

*Напишите ниже 6–10 предложений.*

In [ ]:
# Напишите здесь ваш текстовый ответ (можно оставить как комментарий)
# ...


In [ ]:
# ================================
# ЯЧЕЙКА 5. Подготовка данных: масштабирование
# ================================
X = df.values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Готово: данные масштабированы (StandardScaler).")


In [ ]:
# ================================
# ЯЧЕЙКА 6. PCA для визуализации в 2D
# ================================
pca2 = PCA(n_components=2, random_state=42)
X_pca2 = pca2.fit_transform(X_scaled)

print("Доля объяснённой дисперсии (2 компоненты):", np.round(pca2.explained_variance_ratio_.sum(), 4))

plt.figure()
plt.scatter(X_pca2[:,0], X_pca2[:,1], s=10)
plt.title("PCA (2D) — исходные точки")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.show()


## Ячейка 7. Кластеризация KMeans

1) Подберите число кластеров `k` (например, 2–8).  
2) Для каждого `k` вычислите **silhouette score** (коэффициент силуэта).  
3) Выберите `k` с наилучшим (или близким к лучшему) значением и объясните выбор.

> Silhouette score находится в диапазоне примерно от -1 до 1. Чем больше — тем лучше разделены кластеры (в среднем).


In [ ]:
# ================================
# ЯЧЕЙКА 7.1. Подбор k по silhouette score
# ================================
k_values = list(range(2, 9))
scores = []

for k in k_values:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    scores.append(score)

pd.DataFrame({"k": k_values, "silhouette": scores})


In [ ]:
# ================================
# ЯЧЕЙКА 7.2. Визуализация silhouette по k
# ================================
plt.figure()
plt.plot(k_values, scores, marker='o')
plt.title("Silhouette score для разных k (KMeans)")
plt.xlabel("k")
plt.ylabel("silhouette score")
plt.tight_layout()
plt.show()

best_k = k_values[int(np.argmax(scores))]
print("Лучший k по silhouette:", best_k)


In [ ]:
# ================================
# ЯЧЕЙКА 7.3. KMeans с выбранным k и визуализация кластеров в PCA-2D
# ================================
k = best_k  # при желании можно заменить вручную
kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
labels_km = kmeans.fit_predict(X_scaled)

plt.figure()
plt.scatter(X_pca2[:,0], X_pca2[:,1], c=labels_km, s=10)
plt.title(f"KMeans (k={k}) — кластеры в PCA-2D")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.show()

print("Размеры кластеров:")
pd.Series(labels_km).value_counts().sort_index()


## Ячейка 8. Кластеризация DBSCAN (для сложных форм и выбросов)

DBSCAN — алгоритм, который умеет находить кластеры произвольной формы и выделять шумовые точки.
- `eps` — радиус окрестности;
- `min_samples` — минимальное число точек в окрестности.

Подберите параметры так, чтобы:
- число кластеров было разумным;
- была выделена часть шумовых точек (метка -1), если они есть.


In [ ]:
# ================================
# ЯЧЕЙКА 8.1. DBSCAN: настройка параметров
# ================================
eps = 0.7        # попробуйте 0.3–1.2
min_samples = 8  # попробуйте 5–15

db = DBSCAN(eps=eps, min_samples=min_samples)
labels_db = db.fit_predict(X_scaled)

n_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise = int((labels_db == -1).sum())

print("DBSCAN clusters:", n_clusters)
print("DBSCAN noise points:", n_noise)

plt.figure()
plt.scatter(X_pca2[:,0], X_pca2[:,1], c=labels_db, s=10)
plt.title(f"DBSCAN (eps={eps}, min_samples={min_samples}) — кластеры/шум в PCA-2D")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.show()


## Ячейка 9. Поиск аномалий (дополнительно) — Isolation Forest

Isolation Forest (Изоляционный лес) — алгоритм, который пытается “изолировать” редкие точки быстрее, чем обычные. Подходит для вариантов 10–12 (датчики + выбросы), но можно попробовать и на других.

- `contamination` — ожидаемая доля аномалий (например, 0.03–0.10).


In [ ]:
# ================================
# ЯЧЕЙКА 9.1. Isolation Forest
# ================================
contamination = 0.06  # попробуйте 0.03–0.12

iso = IsolationForest(random_state=42, contamination=contamination)
anom_flag = iso.fit_predict(X_scaled)   # -1 = аномалия, 1 = нормальная точка

n_anom = int((anom_flag == -1).sum())
print("Найдено аномалий:", n_anom)

plt.figure()
plt.scatter(X_pca2[:,0], X_pca2[:,1], s=10)
plt.scatter(X_pca2[anom_flag==-1,0], X_pca2[anom_flag==-1,1], s=20)
plt.title("PCA-2D: найденные аномалии (Isolation Forest)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.show()


## Ячейка 10. Анализ результатов (заполнить вручную)

Напишите развёрнутые ответы:
1) Какие кластеры вы получили? Сколько кластеров кажется содержательным и почему?  
2) Как меняется результат при использовании KMeans vs DBSCAN?  
3) Если есть аномалии: как вы их интерпретируете?  
4) Какие признаки, по вашему мнению, сильнее всего влияют на сегментацию/структуру?

*Напишите 10–15 предложений.*

In [ ]:
# Напишите здесь ваш текстовый ответ (можно оставить как комментарий)
# ...


## Ячейка 11. Выводы

Сформулируйте выводы по ПЗ-6:
- что удалось выявить в данных;
- какие методы оказались наиболее подходящими;
- что можно улучшить (например, собрать больше данных, добавить признаки, попробовать другие алгоритмы).

*Не менее 8–10 предложений.*

In [ ]:
# Напишите здесь выводы (можно оставить как комментарий)
# ...
